# 06 — Transformasi Fourier 2D, Domain Frekuensi, dan Analisis Spektral Citra

### Dari variasi intensitas menuju struktur spektral dan besaran fisik yang dapat diukur

Pada notebook 01, citra dipahami sebagai hasil proses akuisisi yang mengubah informasi cahaya menjadi angka. Notebook 02 mempelajari **distribusi nilai** intensitas, tanpa memperhitungkan letak setiap piksel. Notebook 03 menunjukkan bahwa hubungan antarpiksel dapat dimodelkan melalui konvolusi; notebook 04 dan 05 mengembangkan deteksi batas serta segmentasi objek. Kini kita beralih ke pertanyaan berbeda: **pola spasial dengan panjang skala tertentu muncul bagaimana dalam spektrum, dan apa yang berubah ketika citra mengalami blur, noise, sampling, atau filtering?**

Transformasi Fourier bukan alat ajaib untuk melihat detail tersembunyi. Ia menyatakan citra dalam basis gelombang kompleks. Interpretasi spektrum harus mempertimbangkan **satuan, fase, tepi citra, batas sampling, model sensor, serta kecocokan asumsi linearitas dan invariansi pergeseran (LSI)**. Spektrum yang tajam tidak otomatis berarti pengukuran yang akurat.

> **Kontrak pembelajaran.** Notebook ini independen dan dapat dijalankan dari atas ke bawah. `data.camera()` dipakai sebagai ilustrasi citra nyata bawaan pustaka; studi kasus akhir menggunakan **data sintetis dengan ground truth diketahui** dan *pixel pitch* asumsi, bukan foto atau kalibrasi perangkat fisik. Semua angka kinerja hanya berlaku pada simulator yang dijelaskan.

## Peta kompetensi dan pertanyaan pemandu

Setelah menyelesaikan materi, pembaca diharapkan mampu:

1. Menurunkan DFT dua dimensi dari ekspansi basis kompleks dan menjelaskan *spatial frequency* beserta satuannya.
2. Menghubungkan amplitudo, fase, sifat simetri, energi spektral, dan rekonstruksi.
3. Menjelaskan kebocoran spektral (*spectral leakage*), *windowing*, dan artefak pemotongan citra.
4. Membedakan konvolusi **linear** dari konvolusi **sirkular** yang tersirat oleh DFT diskret.
5. Menurunkan hubungan PSF, OTF, MTF, blur Gaussian, dan respons sistem pencitraan di domain frekuensi.
6. Merancang filter low-pass, high-pass, band-pass, dan notch sambil membahas ringing, noise, dan *information loss*.
7. Menguji pergeseran citra, aliasing, dan efek skala piksel anisotropik melalui eksperimen.
8. Melakukan eksperimen berbasis dataset terpisah dengan PSNR, SSIM, galat ukuran fisik, waktu proses, dan bootstrap tingkat spesimen.

**Pertanyaan diagnostik.** Apakah spektrum yang lebih terang berarti citra lebih tajam? Mengapa dua citra yang magnitudo FFT-nya sama bisa tampak berbeda? Apa yang terjadi jika citra tiba-tiba diperlakukan periodik pada batasnya? Apakah frekuensi `0,1 cycle/pixel` sama dengan `0,1 cycle/mm`?

## 1. Kontrak numerik: indeks, satuan, dan dua jenis frekuensi

Kita menggunakan citra $I[n,m]$ dengan $n$ sebagai **indeks baris** (bertambah ke bawah) dan $m$ indeks kolom (bertambah ke kanan). Ukuran citra adalah $M\times N$ piksel, sedangkan *pixel pitch* adalah $p_y$ mm/piksel pada arah baris dan $p_x$ mm/piksel pada arah kolom. Koordinat fisik adalah $y=n p_y$ dan $x=m p_x$.

Frekuensi indeks DFT $k$ dan $\ell$ tidak memiliki satuan. Setelah normalisasi menjadi $k/M$ dan $\ell/N$, satuannya **cycles/pixel**; setelah dibagi pitch menjadi $f_y=k/(Mp_y)$ dan $f_x=\ell/(Np_x)$, satuannya **cycles/mm**. Panjang perioda fisik sepanjang arah normal garis fase adalah

$$
P_{\mathrm{mm}}=\frac{1}{\sqrt{(f_{x,\mathrm{pix}}/p_x)^2+(f_{y,\mathrm{pix}}/p_y)^2}}.
$$

Rumus di atas berlaku untuk pola sinusoidal dua dimensi yang diberi frekuensi $f_{x,\mathrm{pix}},f_{y,\mathrm{pix}}$. **Jangan** mengalikan perioda piksel dengan satu pitch ketika $p_x\ne p_y$ dan pola miring.

In [ ]:
import platform, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage as ndi, signal
from skimage import data, img_as_float, metrics

SEED=20261112
rng=np.random.default_rng(SEED)
plt.rcParams.update({'figure.figsize':(10,4),'axes.grid':False,'figure.dpi':105})

camera=img_as_float(data.camera())
assert camera.ndim == 2 and 0 <= camera.min() and camera.max() <= 1
print('Python:',platform.python_version(),'| NumPy:',np.__version__,'| citra:',camera.shape)
print('Citra referensi: skimage.data.camera(); data didistribusikan sebagai contoh pustaka.')

## 2. DFT sebagai proyeksi ke basis gelombang kompleks

Deret Fourier untuk sinyal periodik merepresentasikan fungsi sebagai superposisi komponen harmonik. Untuk citra diskret, kita memakai basis $\exp[-j2\pi(kn/M+\ell m/N)]$, yang terpisah di kedua sumbu. Dengan $j^2=-1$, transformasi dan inversnya adalah

$$
F[k,\ell]=\sum_{n=0}^{M-1}\sum_{m=0}^{N-1} I[n,m]\,e^{-j2\pi(kn/M+\ell m/N)},
$$

$$
I[n,m]=\frac{1}{MN}\sum_{k=0}^{M-1}\sum_{\ell=0}^{N-1}F[k,\ell]\,e^{j2\pi(kn/M+\ell m/N)}.
$$

**Mengapa dapat dibalik?** Karena fungsi basis kompleks ortogonal pada grid diskret: $\sum_{n=0}^{M-1}e^{j2\pi(k-k')n/M}=M\,\delta_{kk'}$ untuk indeks frekuensi yang berbeda modulo $M$. Ortogonalitas 2D merupakan produk dari dua ortogonalitas 1D. Faktor $1/(MN)$ mengikuti konvensi normalisasi NumPy: `fft2` tidak menormalisasi transformasi maju.

Koefisien $F[0,0]=\sum I[n,m]$ adalah komponen DC, bukan nilai rata-rata itu sendiri. Rata-ratanya adalah $F[0,0]/(MN)$. `fftshift` hanya memindahkan posisi frekuensi nol ke tengah **untuk visualisasi**; tidak mengubah sinyal fisik.

In [ ]:
# DFT langsung pada array kecil: benar tetapi O(M^2 N^2), sehingga hanya untuk validasi.
small=np.array([[1.,2.,0.],[2.,1.,3.],[0.,3.,2.]])
M0,N0=small.shape
n,m=np.indices(small.shape)
Fmanual=np.zeros(small.shape, dtype=complex)
for k in range(M0):
    for ell in range(N0):
        Fmanual[k,ell]=np.sum(small*np.exp(-2j*np.pi*(k*n/M0+ell*m/N0)))
Ffft=np.fft.fft2(small)
assert np.allclose(Fmanual,Ffft,atol=1e-12)
assert np.allclose(np.fft.ifft2(Ffft).real,small,atol=1e-12)
assert np.isclose(Ffft[0,0].real,small.sum())
print('Maksimum beda implementasi DFT vs FFT:',float(np.max(np.abs(Fmanual-Ffft))))
print('DC:',Ffft[0,0],'; rerata intensitas:',Ffft[0,0].real/small.size)

## 3. Apa arti puncak frekuensi dalam gambar?

Pola sinusoidal $I[n,m]=a+b\cos[2\pi(f_xm+f_yn)+\phi]$ menghasilkan dua puncak spektral kompleks pada frekuensi berlawanan, selain komponen DC (untuk frekuensi yang tepat terletak pada bin DFT). Besarnya vektor frekuensi $f=\sqrt{f_x^2+f_y^2}$ berhubungan dengan **perioda normal terhadap garis fase** $P=1/f$, bukan otomatis jarak horizontal antar-garis.

Pola diagonal memberi frekuensi $f_x$ dan $f_y$ sekaligus. Posisi puncak menyatakan skala dan orientasi; amplitudo puncak terkait kontras; fase koefisien bergantung pada translasi pola. Karena citra bernilai real, $F[-k,-\ell]=F[k,\ell]^*$ dengan indeks periodik: pasangan frekuensi ini adalah **simetri konjugat**, bukan dua objek terpisah.

In [ ]:
H,W=192,224
Y,X=np.mgrid[:H,:W]
fx,fy=14/W,9/H  # tepat pada bin FFT
pattern=.45+.25*np.cos(2*np.pi*(fx*X+fy*Y)+.8)
Fs=np.fft.fftshift(np.fft.fft2(pattern))
fy_axis=np.fft.fftshift(np.fft.fftfreq(H))
fx_axis=np.fft.fftshift(np.fft.fftfreq(W))
fig,axs=plt.subplots(1,3,figsize=(13,3.7))
axs[0].imshow(pattern,cmap='gray',vmin=0,vmax=1);axs[0].set_title('Pola dengan frekuensi diketahui')
axs[1].imshow(np.log1p(np.abs(Fs)),origin='lower',extent=[fx_axis[0],fx_axis[-1],fy_axis[0],fy_axis[-1]],cmap='magma')
axs[1].set(title='Log-magnitudo FFT (2D)',xlabel='Frekuensi $f_x$ (cycles/pixel)',ylabel='$f_y$ (cycles/pixel)')
axs[2].plot(fx_axis,np.abs(Fs[H//2+9,:]),label='Penampang $f_y$ positif')
axs[2].set(title='Posisi puncak pada penampang',xlabel='$f_x$ (cycles/pixel)',ylabel='Amplitudo FFT')
axs[2].legend()
axs[0].axis('off');fig.tight_layout();plt.show()
print('Perioda normal:',round(1/np.hypot(fx,fy),3),'piksel')

## 4. Mengapa fase tidak boleh diabaikan?

Tuliskan spektrum sebagai $F[k,\ell]=|F[k,\ell]|e^{j\varphi[k,\ell]}$. Magnitudo menyatakan bobot komponen, sedangkan fase mengodekan **penempatan relatif** struktur. Pergeseran citra secara sirkular sebesar $(\Delta n,\Delta m)$ mengubah spektrumnya menurut

$$
F_{\mathrm{geser}}[k,\ell]=F[k,\ell]\,e^{-j2\pi(k\Delta n/M+\ell\Delta m/N)}.
$$

Magnitudo tidak berubah, tetapi fase memperoleh ramp linear. Dua citra yang memiliki magnitudo identik karena geseran dapat memiliki letak objek berbeda. Contoh di bawah menunjukkan bahwa menyebut spektrum sebagai *tekstur* tanpa menganalisis fase dapat menghilangkan informasi geometri.

In [ ]:
patch=camera[80:272,100:324]
shifted=np.roll(patch,shift=(13,-21),axis=(0,1))
P=np.fft.fft2(patch);Q=np.fft.fft2(shifted)
assert np.allclose(np.abs(P),np.abs(Q),rtol=1e-11,atol=1e-8)
fig,ax=plt.subplots(1,3,figsize=(11,3.2))
for a,z,t in zip(ax,[patch,shifted,shifted-patch],['Asli','Geser sirkular','Selisih spasial']):
    a.imshow(z,cmap='gray');a.set_title(t);a.axis('off')
plt.tight_layout();plt.show()
print('Beda magnitudo maksimum:',np.max(np.abs(np.abs(P)-np.abs(Q))))

## 5. Parseval: energi tidak menghilang ketika mengganti basis

Dengan konvensi FFT NumPy, ortogonalitas basis menghasilkan persamaan Parseval:

$$
\sum_{n,m}|I[n,m]|^2=\frac{1}{MN}\sum_{k,\ell}|F[k,\ell]|^2.
$$

Ini adalah hukum konservasi energi matematis pada transformasi unitary-tereskalakan, **bukan klaim konservasi energi foton** pada proses pencitraan. Kuadrat magnitudo $|F|^2$ disebut *periodogram* mentah, tetapi interpretasinya sebagai estimasi *power spectral density* (PSD) menuntut normalisasi, definisi sampling, dan perhatian pada bias–variance. Magnitudo tampilan `log1p(abs(F))` bukan PSD.

In [ ]:
Espace=np.sum(np.abs(patch)**2)
Efreq=np.sum(np.abs(np.fft.fft2(patch))**2)/patch.size
assert np.isclose(Espace,Efreq,rtol=1e-12)
print('Energi spasial:',round(float(Espace),6),'| energi spektral:',round(float(Efreq),6))
# Peta log skala berbeda dari energi spektral
A=np.fft.fftshift(np.abs(np.fft.fft2(patch)))
fig,axs=plt.subplots(1,2,figsize=(9,3.8))
axs[0].imshow(np.log1p(A),cmap='magma');axs[0].set_title('log(1+|F|) untuk tampilan')
axs[1].imshow(np.log1p(A**2),cmap='inferno');axs[1].set_title('log(1+|F|²), tidak dinormalisasi')
for a in axs:a.axis('off')
plt.tight_layout();plt.show()

## 6. Spectral leakage, bin FFT, dan pemotongan jendela observasi

DFT mengasumsikan sinyal yang diamati akan **diulang periodik**. Jika ada diskontinuitas antara ujung kiri dan kanan bidang pengamatan, energi sinyal yang semestinya terkonsentrasi dapat tersebar ke banyak bin. Perkalian sinyal dengan jendela $w[n,m]$ di ruang berdampak pada **konvolusi spektrum** dengan $W[k,\ell]$, sehingga jendela menekan sidelobe tetapi **memperlebar mainlobe**.

Dengan Hann 2D terpisahkan, $w[n,m]=w_y[n]w_x[m]$. Penggunaan Hann membantu deteksi puncak pada pola yang tidak pas dengan bin FFT, tetapi harus dikoreksi gain-nya jika amplitudo ingin diukur secara kuantitatif. Bandwidth yang lebih lebar menyebabkan dua puncak berdekatan semakin sulit dipisahkan. Tidak ada jendela yang menghilangkan seluruh trade-off.

In [ ]:
N=256
xx=np.arange(N)
f0=14.38/N
z=np.cos(2*np.pi*f0*xx)
w=np.hanning(N)
Sf=np.fft.rfft(z); Sw=np.fft.rfft(z*w)
fq=np.fft.rfftfreq(N)
fig,ax=plt.subplots(figsize=(10,3.5))
ax.plot(fq,20*np.log10(np.maximum(np.abs(Sf)/np.max(abs(Sf)),1e-6)),label='Jendela persegi (tanpa window)')
ax.plot(fq,20*np.log10(np.maximum(np.abs(Sw)/np.max(abs(Sw)),1e-6)),label='Hann')
ax.set(xlabel='Frekuensi (cycles/pixel)',ylabel='Magnitudo relatif (dB)',ylim=(-100,5),xlim=(0,.17),title='Leakage dan trade-off jendela')
ax.legend();plt.tight_layout();plt.show()
print('Frekuensi sebenarnya:',f0,'cycles/pixel; resolusi bin:',1/N)

## 7. Grid frekuensi: frekuensi positif, negatif, dan pitch anisotropik

Untuk dimensi $N$, `np.fft.fftfreq(N, d=p_x)` menghasilkan frekuensi dalam cycles/mm apabila `d` dinyatakan dalam mm. Untuk indeks frekuensi di atas frekuensi Nyquist, representasi signed-frequency muncul sebagai bilangan negatif. `fftshift` mengurutkan frekuensi negatif ke kiri/atas dan positif ke kanan/bawah.

Batas Nyquist setiap sumbu adalah $|f_x|\le 1/(2p_x)$ dan $|f_y|\le 1/(2p_y)$, bukan lingkaran Nyquist tunggal dalam satuan fisik. Jika pitch anisotropik, bentuk domain frekuensi fisik dan panjang gelombang efektif bergantung arah. Nilai `cycles/pixel` hanya menyatakan sampel, bukan dimensi objek.

In [ ]:
px,py=.012,.018  # mm per piksel; asumsi demonstrasi
fy_p=np.fft.fftshift(np.fft.fftfreq(H,d=py))
fx_p=np.fft.fftshift(np.fft.fftfreq(W,d=px))
period_from_phys=1/np.hypot(fx/px,fy/py)
period_wrong=(1/np.hypot(fx,fy))*px
print(f'Pitch x/y: {px:.3f}/{py:.3f} mm/px')
print(f'Perioda fisik sesuai 2D: {period_from_phys:.4f} mm; perkalian naif dengan pitch-x: {period_wrong:.4f} mm')
assert abs(period_from_phys-period_wrong)>1e-4
fig,ax=plt.subplots(figsize=(6,3.8))
ax.imshow(np.log1p(np.abs(Fs)),origin='lower',extent=[fx_p[0],fx_p[-1],fy_p[0],fy_p[-1]],cmap='magma')
ax.set(xlabel='$f_x$ (cycles/mm)',ylabel='$f_y$ (cycles/mm)',title='Spektrum dalam satuan fisik, pitch anisotropik')
plt.tight_layout();plt.show()

## 8. Teorema konvolusi dan sumber kesalahan circular boundary

Bila $g=h*f$ adalah konvolusi **sirkular** dua citra dengan ukuran sama, DFT memberikan $G=HF$ secara tepat. Untuk konvolusi **linear** (tanpa *wrap-around*) antara sinyal berukuran $M\times N$ dan kernel $P\times Q$, ukuran domain bebas aliasing adalah setidaknya $(M+P-1)\times(N+Q-1)$ sebelum dicrop. Ketika kernel diperlakukan secara sirkular tanpa padding, objek yang keluar pada batas kanan dapat 'masuk' dari batas kiri.

$$
\mathcal F\{f\circledast h\}[k,\ell]=F[k,\ell]H[k,\ell],
$$

sedangkan konvolusi linear $f*h$ perlu padding atau implementasi khusus. Simbol $\circledast$ membedakan konvolusi sirkular dari $*$ linear. *Boundary mode* `reflect`, `constant`, dan `wrap` memiliki asumsi fisik berbeda; tidak satu pun boleh dipilih semata-mata karena hasilnya tampak paling halus.

In [ ]:
A=np.zeros((61,81));A[6:15,3:12]=1
ker=np.ones((15,17),float)/(15*17)
# scipy.signal.fftconvolve menghasilkan konvolusi linear dengan padding penuh lalu crop 'same'
linear=signal.fftconvolve(A,ker,mode='same')
# Hanya padding kernel (roll) untuk alignment pusat pada sirkular
circular=np.fft.ifft2(np.fft.fft2(A)*np.fft.fft2(ker,s=A.shape)).real
circular=np.roll(circular,(-ker.shape[0]//2+1,-ker.shape[1]//2+1),(0,1))
# Validasi jauh dari tepi menggunakan sinyal titik interior
inter=np.zeros((61,81));inter[31,41]=1
l2=signal.fftconvolve(inter,ker,mode='same')
c2=ndi.convolve(inter,ker,mode='wrap')
assert np.allclose(l2,c2,atol=1e-12)
fig,ax=plt.subplots(1,3,figsize=(11,3))
for a,v,t in zip(ax,[A,linear,circular],['Input dekat batas','Konvolusi linear: zero padding','Konvolusi circular: wrap']):
    a.imshow(v,cmap='gray');a.set_title(t);a.axis('off')
plt.tight_layout();plt.show()
print('Perbedaan boundary (L1):',float(np.abs(linear-circular).sum()))

## 9. LSI, PSF, OTF, dan MTF: model optik yang dapat diuji

Untuk sistem linear yang responsnya tidak berubah ketika objek digeser (*shift-invariant*), citra ideal $f$ diubah menjadi $g=h*f+\eta$, dengan $h$ sebagai *point spread function* (PSF), $\eta$ sebagai noise, dan integral PSF dinormalisasi menjadi satu agar intensitas rata-rata terjaga. Transformasi Fourier memberi

$$
G(f_x,f_y)=H(f_x,f_y)F(f_x,f_y)+N(f_x,f_y),
$$

$$
\operatorname{OTF}(f_x,f_y)=\mathcal F\{h\}(f_x,f_y),\qquad
\operatorname{MTF}(f_x,f_y)=\frac{|\operatorname{OTF}(f_x,f_y)|}{|\operatorname{OTF}(0,0)|}.
$$

MTF adalah **magnitudo** OTF (dinormalisasi); fase OTF disebut PTF. Model ini cocok untuk pencitraan intensitas *incoherent* yang dimodelkan linear terhadap irradiance pada kondisi tetap. Untuk pencitraan *coherent*, hubungan gelombang kompleks dan transfer amplitudo harus ditangani berbeda; pembagian OTF menjadi nilai nol atau hampir nol juga menyebabkan rekonstruksi invers tidak stabil.

In [ ]:
# PSF Gaussian terpusat, MTF diskret, dan pemeriksaan model analitis.
Npsf=161
q=np.arange(Npsf)-Npsf//2
Yp,Xp=np.meshgrid(q,q,indexing='ij')
sigma_psf=2.2
psf=np.exp(-(Xp**2+Yp**2)/(2*sigma_psf**2));psf/=psf.sum()
OTF=np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(psf)))
MTF=np.abs(OTF);MTF/=MTF.max()
fp=np.fft.fftshift(np.fft.fftfreq(Npsf))
mtf_theory=np.exp(-2*np.pi**2*sigma_psf**2*fp**2)
assert np.allclose(MTF[Npsf//2,:],mtf_theory,atol=1e-10)
fig,ax=plt.subplots(1,3,figsize=(12,3.5))
ax[0].imshow(psf,cmap='viridis');ax[0].set_title('PSF Gaussian, integral = 1')
ax[1].imshow(MTF,cmap='magma',vmin=0,vmax=1);ax[1].set_title('MTF dua dimensi')
ax[2].plot(fp,MTF[Npsf//2,:],label='DFT PSF')
ax[2].plot(fp,mtf_theory,'--',label='Analitik')
ax[2].set(xlabel='Frekuensi (cycles/pixel)',ylabel='MTF',title='MTF pada $f_y=0$',xlim=(0,.25));ax[2].legend()
for a in ax[:2]:a.axis('off')
plt.tight_layout();plt.show()

## 10. Gaussian blur dan batas resolusi informasi

Jika PSF spasial Gaussian isotropik dengan simpangan baku $\sigma$ piksel, respons frekuensi kontinu idealnya

$$
H(f_x,f_y)=\exp[-2\pi^2\sigma^2(f_x^2+f_y^2)].
$$

Akibatnya, frekuensi tinggi dilemahkan secara berangsur, **tidak dihentikan mendadak**. Blur bukan sekadar mengurangi varians noise: ia juga mengurangi kontras objek kecil, menggabungkan batas yang berdekatan, dan dapat menggeser keputusan segmentasi. Frekuensi yang telah ditekan sangat kuat tidak dapat dipulihkan andal tanpa asumsi tambahan; inilah pengantar menuju *inverse problems*.

Untuk Gaussian, frekuensi saat MTF=0,5 adalah $f_{50}=\sqrt{\ln 2}/(\sqrt 2\pi\sigma)$ cycles/pixel, sepanjang sumbu mana pun pada model isotropik tersebut.

In [ ]:
for s in [0.8,1.6,3.2]:
    f50=np.sqrt(np.log(2))/(np.sqrt(2)*np.pi*s)
    print(f'sigma={s:.1f} px -> f50 analitik={f50:.4f} cycle/px')
fig,ax=plt.subplots(1,2,figsize=(10,3.5))
ax[0].imshow(ndi.gaussian_filter(camera,sigma=2.4),cmap='gray');ax[0].set_title('Kamera dengan blur Gaussian')
fscan=np.linspace(0,.5,400)
for s in [.8,1.6,3.2]:ax[1].plot(fscan,np.exp(-2*np.pi**2*s*s*fscan*fscan),label=f'σ={s} px')
ax[1].axhline(.5,color='gray',ls=':');ax[1].set(xlabel='Frekuensi (cycles/pixel)',ylabel='MTF',title='Transfer frekuensi Gaussian')
ax[1].legend();ax[0].axis('off');plt.tight_layout();plt.show()

## 11. Filter ideal, Butterworth, Gaussian, dan ringing

Filter low-pass ideal pada radius frekuensi $\rho$ memakai $H(\rho)=1$ jika $\rho\le\rho_c$ dan nol di atasnya. Diskontinuitas tajam spektrum menghasilkan impuls spasial berosilasi dengan sidelobe, sehingga muncul **ringing** di dekat tepi. Filter Butterworth orde $n$ memberikan transisi lebih halus,

$$
H_{\mathrm{BW}}(\rho)=\frac{1}{1+(\rho/\rho_c)^{2n}},
$$

sedangkan Gaussian $H_G(\rho)=\exp[-\rho^2/(2\sigma_f^2)]$ memiliki impuls yang tidak berosilasi pada model ideal. Parameter cutoff hanya bermakna jika **satuan frekuensinya dituliskan**. Filter low-pass dapat membantu noise frekuensi tinggi, tetapi tidak mempertahankan semua tepi maupun ukuran geometris.

In [ ]:
# IFFT untuk low-pass ideal, Butterworth, dan Gaussian pada citra tepi tajam.
L=256
step=np.zeros((L,L));step[:,L//2:]=1
fyf=np.fft.fftfreq(L)[:,None]; fxf=np.fft.fftfreq(L)[None,:]
rho=np.hypot(fxf,fyf)
fc=.08
Hideal=(rho<=fc).astype(float)
Hbw=1/(1+(rho/fc)**(2*3))
Hg=np.exp(-.5*(rho/(fc/np.sqrt(2*np.log(2))))**2)  # H(fc)=0.5
Fstep=np.fft.fft2(step)
outputs={k:np.fft.ifft2(Fstep*H).real for k,H in [('Ideal',Hideal),('Butterworth',Hbw),('Gaussian',Hg)]}
fig,ax=plt.subplots(1,2,figsize=(11,3.4))
for name,v in outputs.items():ax[0].plot(np.arange(70,188),v[L//2,70:188],label=name)
ax[0].plot(np.arange(70,188),step[L//2,70:188],'k--',alpha=.6,label='Tepi ideal')
ax[0].set(xlabel='Kolom piksel',ylabel='Intensitas',title='Ringing di sekitar diskontinuitas');ax[0].legend(fontsize=8)
for name,H in [('Ideal',Hideal),('Butterworth',Hbw),('Gaussian',Hg)]:
    ax[1].plot(np.fft.fftshift(np.fft.fftfreq(L)),np.fft.fftshift(H[L//2*0,:]),label=name)
ax[1].set(xlabel='Frekuensi (cycles/pixel)',ylabel='Gain',title='Penampang transfer pada $f_y=0$');ax[1].legend(fontsize=8)
plt.tight_layout();plt.show()

## 12. High-pass, band-pass, notch, dan konvensi frekuensi negatif

Secara matematis, $H_{HP}=1-H_{LP}$ adalah bentuk sederhana filter high-pass; untuk band-pass diperlukan dua batas frekuensi. **Notch filter** menargetkan frekuensi tertentu (beserta pasangan simetri konjugatnya), misalnya gangguan pita periodik dari pencahayaan atau sensor. Bila hanya satu sisi dari pasangan Fourier citra real yang dimodifikasi, inverse FFT umumnya menghasilkan komponen imajiner; memotong `.real` akan menyembunyikan inkonsistensi model.

Sebuah notch lunak yang menghilangkan dua puncak di sekitar $\pm(f_{x0},f_{y0})$ dapat dibentuk dari dua faktor Gaussian komplementer. Notch yang terlalu luas juga menghapus frekuensi **objek** yang kebetulan berdekatan; memilih notch dari spektrum tanpa kontrol dapat merusak informasi yang ingin diukur.

In [ ]:
# Gangguan pita periodik dan pasangan notch konjugat.
Y,X=np.mgrid[:192,:224]
clean_demo=.55+.13*np.cos(2*np.pi*(.045*X+.032*Y))
banded=clean_demo+.19*np.cos(2*np.pi*.125*Y+.3)

def notch_transfer(shape, fx0=0.,fy0=.125,radius=.012):
    h,w=shape
    fy=np.fft.fftfreq(h)[:,None]
    fx=np.fft.fftfreq(w)[None,:]
    dplus=(fx-fx0)**2+(fy-fy0)**2
    dminus=(fx+fx0)**2+(fy+fy0)**2
    return (1-np.exp(-dplus/(2*radius**2)))*(1-np.exp(-dminus/(2*radius**2)))

Hnotch=notch_transfer(banded.shape)
corrected=np.fft.ifft2(np.fft.fft2(banded)*Hnotch)
assert np.max(np.abs(corrected.imag))<1e-10
fig,ax=plt.subplots(1,4,figsize=(13,3.2))
for a,v,t in zip(ax,[clean_demo,banded,np.fft.fftshift(Hnotch),corrected.real],['Referensi sintetis','Pita gangguan','Transfer notch','Hasil notch']):
    a.imshow(v,cmap='gray');a.set_title(t);a.axis('off')
plt.tight_layout();plt.show()
print('MSE sebelum/sesudah:',np.mean((banded-clean_demo)**2),np.mean((corrected.real-clean_demo)**2))

## 13. Komponen periodik, tren iluminasi, dan pemisahan skala

Dalam banyak citra, perubahan iluminasi lambat berada pada frekuensi rendah; tekstur produk dapat berada pada frekuensi menengah; noise baca putih menyebar lebih merata di rentang frekuensi. Tetapi **frekuensi saling tumpang tindih**: batas objek besar juga mengandung frekuensi rendah, sedangkan gangguan periodik yang kebetulan sama dengan tekstur tidak dapat dipisahkan sempurna hanya dengan band-pass atau notch.

Strategi menekan DC, menghapus tren, atau membagi citra oleh *flat-field* harus disesuaikan dengan besaran target. Menghilangkan tren dapat membantu melihat detail, tetapi dapat pula menghapus variasi intensitas fisik yang bermakna. Contoh berikut memperlihatkan bahwa high-pass bagus untuk *visual contrast*, tetapi tidak mempertahankan radiometri absolute.

In [ ]:
Y,X=np.mgrid[:180,:220]
illum=.23+.44*(X/220)+.08*(Y/180)
detail=.08*np.cos(2*np.pi*(X/11+Y/27))
object_img=illum+detail
lp=ndi.gaussian_filter(object_img,9)
hp=object_img-lp
fig,ax=plt.subplots(1,3,figsize=(11,3))
for a,v,t in zip(ax,[object_img,lp,hp],['Citra observasi','Komponen frekuensi rendah','Residual high-pass']):
    im=a.imshow(v,cmap='gray');a.set_title(t);a.axis('off');fig.colorbar(im,ax=a,shrink=.7)
plt.tight_layout();plt.show()
print('Rata-rata observasi:',object_img.mean(),'| residual:',hp.mean())

## 14. Aliasing: ketika frekuensi fisik lebih cepat daripada sampling

Sinyal kontinu $\cos(2\pi f x)$ yang disampel pada pitch $p$ tidak bisa dibedakan dari sinyal lain dengan $f'=f+n/p$ pada titik sampel. Nyquist menuntut spektrum yang hendak dibedakan berada di bawah $1/(2p)$ pada masing-masing sumbu. Jika optik atau sampling tidak meredam frekuensi di atas batas ini, **aliasing** menghasilkan struktur periodik semu dan estimasi frekuensi yang salah.

Downsampling tanpa antialias filtering berisiko tinggi. Gaussian pra-filter dapat membantu tetapi bukan filter brick-wall sempurna; pembahasan kualitas pemfilteran harus mempertimbangkan lebar transisi frekuensi, bukan sekadar visualisasi keluaran.

In [ ]:
# Aliasing 1D yang terlihat sebagai grating 2D
N=256
xx=np.arange(N)
ftrue=.61  # > Nyquist 0.5 cycle/pixel
za=np.cos(2*np.pi*ftrue*xx)
falias=abs(ftrue-1)
zb=np.cos(2*np.pi*falias*xx)
assert np.allclose(za,zb,atol=1e-12)
fig,ax=plt.subplots(1,2,figsize=(10,3.3))
ax[0].plot(xx[:45],za[:45],'o-',label=f'Sampled $f={ftrue}$')
ax[0].plot(xx[:45],zb[:45],'--',label=f'Alias $f={falias:.2f}$')
ax[0].set(xlabel='Piksel',ylabel='Intensitas',title='Frekuensi berbeda, sampel sama');ax[0].legend(fontsize=8)
img_high=np.tile(za,(48,1))
ax[1].imshow(img_high,cmap='gray',aspect='auto');ax[1].set_title('Pola alias dua dimensi');ax[1].axis('off')
plt.tight_layout();plt.show()

## 15. Spektrum noise: putih, berwarna, dan akibat filtering

Untuk noise aditif putih independen dengan varians $\sigma_\eta^2$, PSD teoretisnya datar sebelum normalisasi tertentu; untuk noise yang sudah dikonvolusi kernel $h$, PSD keluarannya dikalikan $|H|^2$. Namun **noise Poisson (shot noise)** tidak memiliki varians konstan: pada model foton terhitung, varians jumlah foton sama dengan nilai harapannya, sehingga PSD suatu citra dapat tercampur dengan struktur sinyal, clipping, dan nonstationarity.

Untuk filter spasial linear berbobot $w_i$ pada noise independen dengan varians seragam $\sigma_\eta^2$, varians keluaran di satu titik adalah $\sigma_\eta^2\sum_i w_i^2$. Hubungan ini memeriksa bahwa pengurangan noise bukan *informasi gratis*: pelemahan frekuensi tertentu juga memengaruhi sinyal. Filter median, bilateral, dan threshold spektral nonlinier memerlukan pendekatan analisis berbeda.

In [ ]:
# Uji varians noise putih setelah filter Gaussian: teori bobot-kuadrat vs simulasi.
rng_noise=np.random.default_rng(SEED+5)
noise=rng_noise.normal(0,.08,size=(600,620))
sg=1.6
filtered_noise=ndi.gaussian_filter(noise,sg,mode='reflect')
ker1=ndi.gaussian_filter(np.eye(1,41,20)[0],sg,mode='constant') # kernel 1-D dengan pusat
# var 2D = var * (sum_i w_i^2)^2 untuk kernel Gaussian terpisahkan
expected=(.08**2)*(np.sum(ker1**2)**2)
observed=np.var(filtered_noise[15:-15,15:-15])
assert abs(observed/expected-1)<.07
print(f'Varian teoritis: {expected:.7f}, varian simulasi: {observed:.7f}, rasio: {observed/expected:.3f}')
fig,ax=plt.subplots(1,2,figsize=(9,3.2))
ax[0].imshow(noise[:120,:160],cmap='gray',vmin=-.2,vmax=.2);ax[0].set_title('Noise putih')
ax[1].imshow(filtered_noise[:120,:160],cmap='gray',vmin=-.2,vmax=.2);ax[1].set_title('Setelah filter Gaussian')
for a in ax:a.axis('off')
plt.tight_layout();plt.show()

## 16. Wiener filtering: batas antara filtering dan rekonstruksi invers

Jika model pengamatan $G=HF+N$ diketahui dan sinyal/noise dianggap proses acak dengan PSD masing-masing $S_F,S_N$, solusi Wiener stasioner dapat ditulis

$$
\widehat F=\frac{H^*}{|H|^2+S_N/S_F}\,G.
$$

Bentuk ini merupakan estimasi kuadrat-galat minimum **di bawah asumsi model stokastik tertentu**. Jika $S_F$ atau $S_N$ tidak diketahui, praktik sering menggantinya dengan parameter regulasi efektif. Membagi langsung $G/H$ akan memperbesar noise di tempat $H$ nyaris nol. Kita hanya memperkenalkannya sebagai jembatan menuju notebook *inverse problems*, bukan sebagai jaminan pemulihan detail yang hilang.

In [ ]:
# Ilustrasi stabilisasi invers pada 1D: beda penguatan frekuensi tinggi.
fq=np.fft.rfftfreq(300)
H=np.exp(-2*np.pi**2*(2.0**2)*fq**2)
with np.errstate(divide='ignore',invalid='ignore'):
    naive=1/np.maximum(H,1e-7)
regularized=H/(H*H+.025)
fig,ax=plt.subplots(figsize=(9.5,3.4))
ax.semilogy(fq,naive,label='Invers langsung (dengan lantai numerik)')
ax.semilogy(fq,regularized,label='Gain Wiener-regularized, K=0.025')
ax.set(xlabel='Frekuensi (cycles/pixel)',ylabel='Gain efektif (log)',title='Mengapa dekonvolusi membutuhkan regularisasi')
ax.legend();plt.tight_layout();plt.show()

## 17. Fase Fourier dan registrasi citra (pengantar)

Teorema pergeseran memperlihatkan bahwa bila dua gambar berbeda hanya dalam translasi, maka cross-power spectrum ternormalisasi $R=F_1F_2^*/|F_1F_2^*|$ menghasilkan puncak korelasi pada lokasi translasi relatif (dengan konvensi tanda yang bergantung urutan input). Metode ini disebut **phase correlation**. Puncak yang tajam bukan jaminan pergeseran benar: perubahan skala, rotasi, area non-overlap, dan noise dapat menyebabkan ambiguitas.

Praktik yang baik adalah melaporkan *shift* dalam piksel **dan**, bila pitch diketahui, dalam satuan panjang. Pergeseran subpiksel memerlukan interpolasi model puncak dan estimasi ketidakpastian, bukan hanya `argmax`.

In [ ]:
# Phase correlation untuk shift integer sirkular (uji numerik yang tandanya eksplisit).
A=camera[64:320,64:320]
true_shift=(9,-16)
B=np.roll(A,true_shift,axis=(0,1))
F1=np.fft.fft2(A);F2=np.fft.fft2(B)
R=F2*np.conj(F1)
R/=np.maximum(np.abs(R),1e-14)
xcorr=np.fft.ifft2(R).real
peak=np.unravel_index(np.argmax(xcorr),xcorr.shape)
est=tuple(int(i if i<=n//2 else i-n) for i,n in zip(peak,A.shape))
assert est==true_shift
print('Shift aktual (baris, kolom):',true_shift,'| estimasi:',est)
fig,ax=plt.subplots(1,3,figsize=(10,3.2))
for a,z,t in zip(ax,[A,B,np.fft.fftshift(xcorr)],['A','B','Korelasi fase (shifted)']):a.imshow(z,cmap='gray');a.set_title(t);a.axis('off')
plt.tight_layout();plt.show()

## 18. Metrik dan desain eksperimen: apa tujuan filtering?

PSNR memerlukan referensi dan `data_range` benar. Untuk citra intensitas $I$ dan hasil $\widehat I$,

$$
\operatorname{MSE}=\frac{1}{MN}\sum_{n,m}(I[n,m]-\widehat I[n,m])^2,\qquad
\operatorname{PSNR}=10\log_{10}\frac{L^2}{\operatorname{MSE}}.
$$

SSIM menggabungkan luminansi, kontras, dan struktur lokal; ia sensitif terhadap ukuran jendela, `data_range`, serta preprocessing. **Keduanya bukan metrik universal metrologi.** Untuk inspeksi pola periodik, kita juga harus menguji galat perioda **dalam mm**, kestabilan estimasi antarspesimen, serta biaya komputasi.

Eksperimen berikut dirancang untuk menunjukkan trade-off. Tidak ada klaim bahwa metode terbaik di bawah simulator berarti terbaik untuk kamera atau pabrik nyata. Unit inferensi adalah **spesimen**, bukan piksel.

## 19. Studi kasus utama — inspeksi periodisitas permukaan dengan gangguan periodik

Kita membuat *digital phantom* permukaan yang memiliki pola periodik dengan orientasi, fase, frekuensi, dan pencahayaan berbeda. Model forward (versi sederhana):

$$
I_0[n,m]=b[n,m]+A\cos\{2\pi(f_xm+f_yn)+\phi\}+d[n,m],
$$

$$
I_{\mathrm{ref}}=h_{\mathrm{optik}}*I_0,\qquad
I_{\mathrm{obs}}=Q_{12}\!\left[\operatorname{Poisson}\{N_\gamma(I_{\mathrm{ref}}+s)\}/N_\gamma+\epsilon_r\right].
$$

$s[n,m]$ adalah **gangguan periodik sensor** sepanjang arah baris (diperkenalkan setelah blur optik), $Q_{12}$ kuantisasi 12 bit, dan $\epsilon_r$ noise baca Gaussian. Ini **asumsi simulator**, bukan model lengkap kamera nyata. Referensi adalah citra terblur secara optik tanpa gangguan periodik dan tanpa noise acak. Karena PSF diasumsikan diketahui pada simulator, kita dapat memisahkan pertanyaan filtering gangguan dari pertanyaan dekonvolusi optik.

**Kontrak dataset:** 9 spesimen validasi untuk memilih parameter; 14 spesimen pengujian yang tidak digunakan untuk memilih parameter; 6 spesimen domain-stres dengan gangguan lebih kuat untuk memeriksa transfer. Pitch asumsi $p_x=0.012$ dan $p_y=0.018$ mm/piksel, sehingga kesalahan perioda dinilai dalam mm berdasarkan parameter forward.

In [ ]:
PIXEL_PITCH_X=.012
PIXEL_PITCH_Y=.018
SHAPE=(192,224)

def simulate_surface(seed,stress=False):
    rg=np.random.default_rng(seed)
    h,w=SHAPE
    y,x=np.mgrid[:h,:w]
    period_px=rg.uniform(12.5,19.0)
    theta=np.deg2rad(rg.uniform(18,48))
    fx=np.cos(theta)/period_px
    fy=np.sin(theta)/period_px
    phase=rg.uniform(0,2*np.pi)
    base=.46 + .19*np.cos(2*np.pi*(fx*x+fy*y)+phase)
    base+=.06*((x/w)-.5)+.05*((y/h)-.5)
    # Cacat halus/inhomogenitas besar yang tidak ikut label perioda.
    xc,yc=rg.uniform(40,w-40),rg.uniform(35,h-35)
    base-=.065*np.exp(-(((x-xc)/23)**2+((y-yc)/16)**2)/2)
    base=np.clip(base,.05,.94)
    sigma_opt=1.0 if not stress else 1.35
    reference=ndi.gaussian_filter(base,sigma_opt,mode='reflect')
    amp=rg.uniform(.13,.22) if not stress else rg.uniform(.21,.31)
    stripe=.125 if not stress else .125 + rg.uniform(-.006,.006)
    interfered=reference+amp*np.cos(2*np.pi*stripe*y+rg.uniform(0,2*np.pi))
    # Shot noise plus read noise; intensitas dinormalisasi terhadap full well semu.
    fullwell=650 if not stress else 320
    shot=rg.poisson(np.clip(interfered,.01,.98)*fullwell)/fullwell
    noisy=np.clip(shot+rg.normal(0,.025 if not stress else .040,reference.shape),0,1)
    observed=np.round(noisy*4095)/4095
    period_mm=1/np.hypot(fx/PIXEL_PITCH_X,fy/PIXEL_PITCH_Y)
    return {'reference':reference,'observed':observed,'period_mm':period_mm,
            'fx':fx,'fy':fy,'stripe_freq':stripe,'seed':seed,'stress':stress}

val_data=[simulate_surface(SEED+1000+i) for i in range(9)]
test_data=[simulate_surface(SEED+5000+i) for i in range(14)]
stress_data=[simulate_surface(SEED+9000+i,stress=True) for i in range(6)]
assert len({d['seed'] for d in val_data+test_data+stress_data})==29
assert all(v['observed'].shape==SHAPE for v in val_data+test_data)
print('Jumlah spesimen: validasi',len(val_data),'| test',len(test_data),'| stres',len(stress_data))
print('Rentang perioda fisik test:',round(min(d['period_mm'] for d in test_data),4),'s.d.',round(max(d['period_mm'] for d in test_data),4),'mm (asumsi)')

In [ ]:
# Tampilkan ground truth, hasil akuisisi, dan spektrum beserta dua sumber puncak.
d=val_data[0]
fig,ax=plt.subplots(1,4,figsize=(14,3.1))
for a,z,t in zip(ax[:3],[d['reference'],d['observed'],d['observed']-d['reference']],
                 ['Referensi (blur optik saja)','Observasi (pita+noise)','Residual pengamatan']):
    a.imshow(z,cmap='gray',vmin=0 if 'Residual' not in t else None,vmax=1 if 'Residual' not in t else None)
    a.set_title(t);a.axis('off')
Fobs=np.fft.fftshift(np.fft.fft2(d['observed']-np.mean(d['observed'])))
ax[3].imshow(np.log1p(abs(Fobs)),cmap='magma');ax[3].set_title('Spektrum log');ax[3].axis('off')
plt.tight_layout();plt.show()
print('Perioda benar (simulator):',round(d['period_mm'],5),'mm; frekuensi gangguan:',d['stripe_freq'],'cycle/pixel')

## 20. Estimasi perioda 2D tanpa menebak dari grafik

Untuk estimasi frekuensi pola utama, kita menghitung FFT citra yang dikurangi nilai rata-ratanya lalu dikalikan jendela Hann 2D. Puncak dicari di daerah frekuensi **yang telah ditentukan** $0.03<\sqrt{f_x^2+f_y^2}<0.12$ cycles/pixel, $f_x>0.03$ cycles/pixel. Batas ini adalah asumsi domain periodisitas eksperimen, sehingga metode dapat gagal untuk pola di luar rentang tersebut.

Puncak diskret diinterpolasi secara parabola di ruang **log-amplitudo** menggunakan tiga bin agar estimasi tidak terbatas pada grid FFT. Ini estimator heuristik untuk pola satu sinus dengan noise moderat, **bukan estimator maksimum likelihood** dan tidak memiliki jaminan unbiased. Setelah mendapatkan $\hat f_x,\hat f_y$, perioda fisik diperkirakan memakai pitch anisotropik. Eksperimen berikut mengukur bias, galat absolut, dan kerentanan terhadap gangguan pada spesimen terpisah.

In [ ]:
def quadratic_peak(logmag, idx,axis):
    i,j=idx
    if axis==0:
        a,b,c=logmag[i-1,j],logmag[i,j],logmag[i+1,j]
    else:
        a,b,c=logmag[i,j-1],logmag[i,j],logmag[i,j+1]
    den=a-2*b+c
    if abs(den)<1e-12:return 0.
    return float(np.clip(.5*(a-c)/den,-.5,.5))

def estimate_period_mm(image, px=PIXEL_PITCH_X, py=PIXEL_PITCH_Y):
    h,w=image.shape
    win=np.hanning(h)[:,None]*np.hanning(w)[None,:]
    centered=(image-image.mean())*win
    A=np.abs(np.fft.fftshift(np.fft.fft2(centered)))
    fy=np.fft.fftshift(np.fft.fftfreq(h))[:,None]
    fx=np.fft.fftshift(np.fft.fftfreq(w))[None,:]
    radius=np.hypot(fx,fy)
    roi=(radius>.03)&(radius<.12)&(fx>.03)&(fy>0)
    weights=np.where(roi,A,-np.inf)
    i,j=np.unravel_index(np.argmax(weights),weights.shape)
    lm=np.log(A+1e-10)
    dy=quadratic_peak(lm,(i,j),0) if 1<i<h-2 else 0.
    dx=quadratic_peak(lm,(i,j),1) if 1<j<w-2 else 0.
    fxhat=(j-w//2+dx)/w
    fyhat=(i-h//2+dy)/h
    return 1/np.hypot(fxhat/px,fyhat/py), (fxhat,fyhat)

for d in val_data[:3]:
    est,vector=estimate_period_mm(d['reference'])
    print(f"seed={d['seed']}: referensi {d['period_mm']:.4f} mm, estimasi {est:.4f} mm; f_hat={vector}")

## 21. Pemilihan filter: validasi saja, tidak menyentuh test

Kandidat yang dibandingkan adalah: **tanpa filtering**, **Gaussian spasial** (`sigma` dicari), dan **notch simetris** (radius dicari). Penguatan filtering dipilih menggunakan rerata PSNR pada **9 spesimen validasi** yang memiliki referensi simulator; seluruh parameter kemudian dibekukan sebelum 14 spesimen pengujian dievaluasi. Ini *oracle-style supervised selection* yang mungkin pada simulator, **tidak tersedia begitu saja pada citra lapangan tanpa referensi**.

Fase gangguan tidak diketahui dan tidak diberikan kepada filter. Namun frekuensi pusat notch $f_y=0.125$ cycles/pixel diketahui dari spesifikasi simulator; keunggulan notch dalam eksperimen ini tidak boleh digeneralisasikan ke gangguan dengan frekuensi atau orientasi acak. Pada stress set, pusat notch sengaja digeser untuk menunjukkan konsekuensi model yang tidak persis cocok.

In [ ]:
def filter_image(image,method,param=None):
    if method=='Mentah':return image.copy()
    if method=='Gaussian':return ndi.gaussian_filter(image,float(param),mode='reflect')
    if method=='Notch':
        H=notch_transfer(image.shape,fx0=0,fy0=.125,radius=float(param))
        rec=np.fft.ifft2(np.fft.fft2(image)*H)
        assert np.max(np.abs(rec.imag))<1e-9
        return np.clip(rec.real,0,1)
    raise ValueError(method)

candidates={'Gaussian':[.5,1.0,1.7,2.4], 'Notch':[.005,.010,.017,.027]}
val_rows=[]
for name,params in candidates.items():
    for param in params:
        scores=[]
        for d in val_data:
            pred=filter_image(d['observed'],name,param)
            scores.append(metrics.peak_signal_noise_ratio(d['reference'],pred,data_range=1))
        val_rows.append({'metode':name,'parameter':param,'PSNR_validasi':np.mean(scores)})
val_table=pd.DataFrame(val_rows)
selected={m:float(g.loc[g['PSNR_validasi'].idxmax(),'parameter']) for m,g in val_table.groupby('metode')}
assert set(selected)=={'Gaussian','Notch'}
print('Parameter terpilih tanpa menggunakan test:',selected)
display(val_table.round(4))

## 22. Uji tertahan: kualitas citra vs kesalahan ukuran

Evaluasi test mengukur **PSNR, SSIM, MAE perioda fisik, dan waktu per spesimen**. Metrik perioda dihitung dari hasil filtering menggunakan estimator yang sama untuk semua metode; tidak ada tuning berdasarkan performa test. PSNR/SSIM bisa meningkat tanpa mengubah galat perioda, sebab estimasi frekuensi terutama dipengaruhi posisi puncak spektral, bukan semua perbedaan intensitas.

Waktu komputasi dipengaruhi hardware, implementasi FFT, dan ukuran citra; angka ini hanya **benchmark lokal deskriptif**, bukan rasio kecepatan universal. SSIM menggunakan rentang intensitas eksplisit `data_range=1`, sesuai normalisasi seluruh citra.

In [ ]:
config=[('Mentah',None),('Gaussian',selected['Gaussian']),('Notch',selected['Notch'])]

def evaluate_collection(ds):
    rows=[]
    for d in ds:
        for method,param in config:
            t0=time.perf_counter()
            pred=filter_image(d['observed'],method,param)
            elapsed=(time.perf_counter()-t0)*1e3
            period,_=estimate_period_mm(pred)
            rows.append({'seed':d['seed'],'metode':method,'parameter':param,
                         'PSNR_dB':metrics.peak_signal_noise_ratio(d['reference'],pred,data_range=1),
                         'SSIM':metrics.structural_similarity(d['reference'],pred,data_range=1),
                         'abs_error_mm':abs(period-d['period_mm']),
                         'signed_error_mm':period-d['period_mm'],
                         'elapsed_ms':elapsed})
    return pd.DataFrame(rows)

res_test=evaluate_collection(test_data)
summary=res_test.groupby('metode').agg(PSNR_dB=('PSNR_dB','mean'),SSIM=('SSIM','mean'),
           MAE_perioda_mm=('abs_error_mm','mean'),bias_perioda_mm=('signed_error_mm','mean'),
           waktu_ms=('elapsed_ms','median')).round(5)
assert len(res_test)==len(test_data)*3
assert res_test.groupby('seed')['metode'].nunique().eq(3).all()
display(summary)

In [ ]:
# Kualitas citra dan sinyal 1D ditampilkan pada spesimen test yang tidak digunakan saat tuning.
d=test_data[2]
results=[filter_image(d['observed'],method,param) for method,param in config]
fig,ax=plt.subplots(2,4,figsize=(14,6.5))
for col,(z,name) in enumerate(zip([d['reference']]+results,['Referensi']+[c[0] for c in config])):
    ax[0,col].imshow(z,cmap='gray',vmin=0,vmax=1);ax[0,col].set_title(name);ax[0,col].axis('off')
    ax[1,col].plot(z[z.shape[0]//2,:]);ax[1,col].set(xlabel='Kolom (px)',ylabel='Intensitas',ylim=(0,1))
plt.tight_layout();plt.show()

## 23. Mengapa performa metrik tidak selalu sejalan?

PSNR menghukum kesalahan intensitas kuadrat. SSIM memberi bobot pada struktur lokal. Estimasi perioda memakai *argmax* spektrum yang sering tetap pada posisi sama meskipun citra berisik. Jadi metode yang memperbaiki PSNR belum tentu meningkatkan akurasi perioda, dan sebaliknya. Ketidaksamaan ini bukan anomali; **fungsi objektif yang berbeda mengukur keberhasilan yang berbeda**.

Pada data nyata, ketidakpastian pitch, perubahan lensa, geometri proyeksi, *rolling shutter*, penempatan objek, dan ketidakpastian referensi turut memengaruhi galat. Hasil test sintetis di sini mengisolasi beberapa sumber kesalahan dengan sengaja. Seluruh angka `mm` berasal dari pitch yang ditentukan simulator dan tidak boleh disebut kalibrasi tertelusur.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,3.6))
method_order=['Mentah','Gaussian','Notch']
ax[0].bar(method_order,[summary.loc[m,'PSNR_dB'] for m in method_order]);ax[0].set(title='PSNR pada test',ylabel='PSNR (dB)')
ax[1].bar(method_order,[summary.loc[m,'MAE_perioda_mm']*1000 for m in method_order]);ax[1].set(title='MAE perioda fisik pada test',ylabel='MAE (µm, pitch asumsi)')
plt.tight_layout();plt.show()

## 24. Ketidakpastian statistik: bootstrap pada unit spesimen

Kita membandingkan **selisih berpasangan** PSNR `Notch − Gaussian` pada 14 spesimen test. Bootstrap dilakukan dengan menarik ulang **spesimen**, bukan piksel, agar korelasi internal sebuah gambar tidak dianggap sampel independen. Interval kepercayaan *percentile* 95% didapat dari kuantil 2,5% dan 97,5% rerata 2.000 sampel ulang.

Hati-hati: interval bootstrap hanya merangkum variasi antarspesimen **dari keluarga simulator yang digunakan**. Interval tersebut tidak mencakup ketidakpastian model fisika, pengujian pada lokasi lain, atau risiko kebocoran statistik akibat pemilihan banyak metode menggunakan data test. Karena test di sini dipakai untuk evaluasi akhir, tidak ada hyperparameter yang diperbarui setelah hasilnya dilihat.

In [ ]:
wide=res_test.pivot(index='seed',columns='metode',values='PSNR_dB').sort_index()
delta=(wide['Notch']-wide['Gaussian']).to_numpy()
rg=np.random.default_rng(SEED+151)
rep=2000
boot=rg.choice(delta,size=(rep,len(delta)),replace=True).mean(axis=1)
ci=np.quantile(boot,[.025,.975]); obs=delta.mean()
assert ci[0]<=ci[1]
print(f'Selisih berpasangan rerata PSNR (Notch - Gaussian): {obs:+.3f} dB')
print(f'95% bootstrap percentile CI: [{ci[0]:+.3f}, {ci[1]:+.3f}] dB; n={len(delta)} spesimen')
fig,ax=plt.subplots(figsize=(8.6,3.4))
ax.hist(boot,bins=30,alpha=.85)
ax.axvline(obs,color='black',ls='--',label='Rerata test')
ax.axvline(ci[0],color='gray',ls=':',label='CI 95%');ax.axvline(ci[1],color='gray',ls=':')
ax.set(xlabel='Selisih rerata PSNR (dB)',ylabel='Jumlah bootstrap',title='Ketidakpastian sampel pada simulator')
ax.legend();plt.tight_layout();plt.show()

## 25. Uji perubahan domain: frekuensi gangguan tidak lagi tepat

Pada *stress set*, frekuensi gangguan banding digeser dari pusat nominal $0.125$ cycle/pixel, blur optik sedikit lebih kuat, serta read noise/shot noise diperbesar. Tidak ada parameter filter yang dituning ulang. Hal ini mendemonstrasikan **model mismatch**: notch yang sempit mungkin gagal ketika frekuensi gangguan bergeser, sedangkan notch yang terlalu lebar menghapus sinyal yang tak seharusnya.

Perbandingan di bawah bersifat ilustratif; enam spesimen terlalu sedikit untuk klaim generalisasi. Stres sintetis juga belum mewakili variasi perangkat, batch, atau kondisi lapangan secara realistis. Gunakan hasil sebagai alasan untuk merancang uji domain shift yang lebih luas, bukan sebagai estimasi probabilitas keberhasilan di lapangan.

In [ ]:
res_stress=evaluate_collection(stress_data)
sum_stress=res_stress.groupby('metode').agg(PSNR_dB=('PSNR_dB','mean'),SSIM=('SSIM','mean'),
                MAE_perioda_mm=('abs_error_mm','mean')).round(5)
print('Uji domain stres tanpa retuning:')
display(sum_stress)
fig,ax=plt.subplots(1,2,figsize=(10.5,3.6))
for i,(metric,label) in enumerate([('PSNR_dB','PSNR (dB)'),('MAE_perioda_mm','MAE perioda (mm)')]):
    yy=np.arange(len(method_order));w=.34
    ax[i].bar(yy-w/2,[summary.loc[m,metric] for m in method_order],width=w,label='Test standar')
    ax[i].bar(yy+w/2,[sum_stress.loc[m,metric] for m in method_order],width=w,label='Domain stres')
    ax[i].set_xticks(yy,method_order);ax[i].set_ylabel(label);ax[i].set_title(label)
ax[0].legend(fontsize=8);plt.tight_layout();plt.show()

## 26. Validasi numerik dan uji batas yang tidak boleh dilompati

Penelitian komputasional yang dapat direproduksi memerlukan uji nilai batas, konservasi energi untuk transformasi, sifat simetri konjugat untuk citra real, hubungan konvolusi, dan konsistensi satuan. Periksa pula bagaimana metode bereaksi terhadap citra konstan, sinusoidal tepat-bin, geseran, dan kasus tanpa noise; sebuah implementasi yang membuat grafik bagus bisa tetap salah tanda, salah skala, atau salah definisi frekuensi.

Uji di bawah bersifat *unit checks* kecil, bukan validasi optik secara menyeluruh. Parameter perangkat dan noise merupakan asumsi demonstrasi. Setiap klaim 'teruji' harus dibatasi pada yang memang dijalankan di lingkungan yang dilaporkan.

In [ ]:
# Pemeriksaan otomatis: satuan, fase, Parseval, conjugate symmetry, transfer Gaussian, dan split.
C=np.random.default_rng(SEED+701).normal(size=(32,40))
FC=np.fft.fft2(C)
assert np.allclose(np.fft.ifft2(FC).real,C,atol=1e-12)
assert np.isclose(np.sum(C**2),np.sum(abs(FC)**2)/C.size,rtol=1e-12)
for k,l in [(2,5),(7,9),(1,3)]:assert np.allclose(FC[-k,-l],np.conj(FC[k,l]),atol=1e-10)
assert np.allclose(np.fft.ifft2(np.fft.fft2(C)*1).real,C,atol=1e-12)
assert not (set(d['seed'] for d in val_data)&set(d['seed'] for d in test_data))
assert not (set(d['seed'] for d in test_data)&set(d['seed'] for d in stress_data))
assert all(np.isfinite(v) for v in summary.to_numpy().ravel())
assert all(d['period_mm']>0 for d in test_data)
print('Uji invariansi, simetri, Parseval, data split, nilai hingga, dan skala perioda: LULUS')

## 27. Panduan interpretasi eksperimen dan batas klaim

Kesimpulan yang sah dari eksperimen ini adalah bahwa **pada satu keluarga simulator**, filtering dapat mengubah kualitas intensitas, struktur lokal, dan akurasi perioda secara berbeda. Metode yang dioptimalkan menggunakan PSNR validasi tidak otomatis optimal untuk galat perioda. Gangguan periodik dengan frekuensi diketahui dapat ditargetkan oleh notch, tetapi notching dapat merusak fitur objek di pita yang sama. MTF dan antialiasing membatasi informasi yang tersedia sebelum algoritma dijalankan.

**Klaim yang tidak boleh dibuat:** bahwa satu filter selalu terbaik untuk seluruh kamera; bahwa angka mm sudah terkalibrasi secara metrologis; bahwa interval bootstrap mencakup semua domain produksi; bahwa dekonvolusi mengembalikan informasi yang lenyap sempurna; atau bahwa spektrum log-magnitudo selalu menunjukkan PSD fisik.

Jika kelak dataset akuisisi nyata tersedia, catat pitch yang ditelusurkan terhadap standar panjang, exposure, optik, posisi objek, kondisi iluminasi, dark/flat frames, dan anotasi atau alat referensi pengukuran. Tetapkan protokol validasi sebelum menguji data lapangan. Untuk penelitian, tingkatkan desain dengan eksperimen berulang lintas perangkat dan domain yang benar-benar independen.

## 28. Latihan analitis dan HOTS

1. **Pembuktian:** mulai dari ortogonalitas akar satuan kompleks, turunkan pasangan DFT–IDFT 2D. Tunjukkan sumber faktor $1/(MN)$ dan uji pada citra $2\times3$.
2. **Satuan:** sebuah pola memiliki $f_x=0.075$ dan $f_y=0.042$ cycles/pixel; pitch $p_x=12$ µm dan $p_y=18$ µm. Hitung perioda normal dalam mm. Bandingkan dengan asumsi pitch isotropik yang salah.
3. **Konvolusi:** buat array $7\times9$ berisi titik dekat batas. Bandingkan `fftconvolve(...,mode="same")` dengan perkalian FFT tanpa padding; jelaskan sumber wrap-around dan resolusinya.
4. **MTF:** turunkan $f_{50}$ dari Gaussian PSF dengan $\sigma=1.7$ piksel. Konversikan ke cycles/mm untuk pitch horizontal 0,015 mm/pixel. Jelaskan kondisi di mana model PSF Gaussian tidak tepat.
5. **Windowing:** gunakan dua sinusoid dengan pemisahan frekuensi makin kecil. Bandingkan resolvabilitas puncak pada rectangular, Hann, dan Blackman; jelaskan trade-off sidelobe–mainlobe.
6. **Aliasing:** desain sinyal 2D dengan frekuensi horizontal melampaui Nyquist tetapi vertikal di bawahnya. Prediksi posisi alias dari persamaan sebelum menjalankan FFT.
7. **Metrologi:** pilih salah satu spesimen studi kasus; lakukan evaluasi ulang ketika ketidakpastian pitch ±1% diasumsikan. Jelaskan mengapa ketidakpastian pitch tidak hilang walaupun PSNR menjadi sangat tinggi.
8. **Robustness:** perluas frekuensi pita menjadi rentang 0.08–0.15 cycles/pixel dan orientasi acak. Uji apakah notch terpilih masih berguna tanpa melakukan retuning di test.
9. **HOTS—desain eksperimen:** rancang prosedur yang membedakan variasi tekstur asli dari interferensi sensor memakai dark/flat references, pergantian iluminasi, dan pengulangan akuisisi; sebutkan unit analisis serta sumber data leakage.
10. **HOTS—batas rekonstruksi:** buat contoh dua tekstur berbeda yang menghasilkan pengamatan hampir sama setelah PSF low-pass dan noise. Jelaskan mengapa solusi invers membutuhkan prior dan mengapa prior dapat menciptakan detail semu.

**Mini-project kelompok (90–120 menit):** rumuskan masalah pengukuran perioda, buat dataset forward sintetis independen, pilih satu tujuan utama (akurasi ukuran, bukan hanya PSNR), tetapkan parameter pada validasi, lakukan test tertahan, hitung interval ketidakpastian tingkat spesimen, dan tulis laporan dengan format *hipotesis → model → rancangan → hasil → interpretasi → keterbatasan → kesimpulan*. Sertakan kode dan random seed.

## 29. Rangkuman, hubungan antarbab, dan referensi

**Gagasan inti:** Fourier adalah representasi ekuivalen dari citra digital, selama sampling dan informasi kompleks (magnitudo serta fase) dipertahankan. Konvolusi menjadi perkalian spektral untuk model LSI, tetapi DFT berukuran terbatas membawa asumsi periodik. OTF/MTF merangkum perilaku transfer optik; filter mengubah distribusi frekuensi dan noise, sedangkan metrik visual tidak selalu menjawab pertanyaan metrologi. Satuan cycles/pixel harus dibedakan dari cycles/mm dan kesalahan kalibrasi tidak disembuhkan oleh transformasi Fourier.

**Koneksi berikutnya:** model PSF/OTF dan dampak blur akan dilanjutkan pada *inverse problems* dan dekonvolusi; estimasi puncak spektral berkaitan dengan ekstraksi fitur dan analisis tekstur; sensitivitas terhadap iluminasi dan warna akan dibahas pada notebook 07. Pengukuran ilmiah yang lebih dalam akan mengembangkan propagasi ketidakpastian, validasi metrologi, *domain shift*, dan inferensi statistik.

**Bacaan utama dan dokumentasi:**

- R. C. Gonzalez dan R. E. Woods, *Digital Image Processing*, ed. ke-4, Pearson, 2018 (DFT, filtering, restorasi).
- J. W. Goodman, *Introduction to Fourier Optics*, ed. ke-4, W. H. Freeman, 2017 (Fourier optics, PSF dan transfer function).
- A. C. Kak dan M. Slaney, *Principles of Computerized Tomographic Imaging*, SIAM, 2001, tersedia daring dari SIAM (invers dan pembentukan citra).
- NumPy Developers, [Discrete Fourier Transform](https://numpy.org/doc/stable/reference/routines.fft.html).
- SciPy Developers, [`scipy.signal.fftconvolve`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.fftconvolve.html).
- scikit-image Developers, [Image quality metrics](https://scikit-image.org/docs/stable/api/skimage.metrics.html).

Dokumen validasi, audit sintaks, dan panduan dosen disediakan bersama paket revisi. Tidak ada data eksternal berbayar yang diperlukan.